# Assignment Beta — Question 1: Closed-Source API vs. Self-Hosted Model for SaafHawa
**Course:** Agentic AI (AI4015), FAST NUCES Karachi  
**Student ID:** 22K-8706  

### Roll-Number Derived Parameters
* Last two digits: $R = 6$
* Digits sum mod 10: $(2+2+8+7+0+6) \bmod 10 = 25 \bmod 10 \implies D = 5$
* Normal daily queries: $Q = 1,500 + 25 \times 6 = \mathbf{1,650}$
* First-attempt LLM calls: $N = 5 + (6 \bmod 4) = \mathbf{7}$
* Tool-call failure probability: $p = 0.04 + 0.01 \times 5 = \mathbf{0.09}\ (9\%)$
* Heatwave days per month: $H = 4 + (6 \bmod 4) = \mathbf{6}$
* Monthly inference budget: $B = 1,200 + 10 \times 6 = \mathbf{\$1,260}$
* End-to-end latency target: $T = 10 - (6 \bmod 3) = \mathbf{10.0\text{ seconds}}$

In [1]:
# Setup parameters
R, D = 6, 5
Q = 1500 + 25 * R
N = 5 + (R % 4)
p = 0.04 + 0.01 * D
H = 4 + (R % 4)
B = 1200 + 10 * R
T = 10 - (R % 3)

print(f"R={R}, D={D} -> Q={Q}, N={N}, p={p:.2f}, H={H}, B=${B}, T={T:.1f}s")

R=6, D=5 -> Q=1650, N=7, p=0.09, H=6, B=$1260, T=10.0s


## 1. Token Volume Calculations
* Calls $k=1 \dots 6$: $\text{Input}_k = 2000 + 800(k-1)$, $\text{Output}_k = 350$, retry probability $p=0.09$
* Call $k=7$: $\text{Input}_7 = 6800$, $\text{Output}_7 = 600$, no retry

In [2]:
# Expected tokens per query
base_inputs = [2000 + 800 * (k - 1) for k in range(1, N + 1)]
base_outputs = [350] * (N - 1) + [600]

exp_inputs = [(1 + p) * inp for inp in base_inputs[:-1]] + [float(base_inputs[-1])]
exp_outputs = [(1 + p) * out for out in base_outputs[:-1]] + [float(base_outputs[-1])]

total_exp_in = sum(exp_inputs)
total_exp_out = sum(exp_outputs)

print(f"Expected Input Tokens/query : {total_exp_in:,.1f}")
print(f"Expected Output Tokens/query: {total_exp_out:,.1f}")

# Monthly volume (30-day month: 24 normal days + 6 heatwave days at 5x Q)
days_normal = 30 - H
days_heatwave = H
q_month = days_normal * Q + days_heatwave * (5 * Q)

monthly_in = q_month * total_exp_in
monthly_out = q_month * total_exp_out

print(f"Monthly Queries             : {q_month:,}")
print(f"Monthly Input Tokens        : {monthly_in:,.0f} ({monthly_in/1e9:.3f} B)")
print(f"Monthly Output Tokens       : {monthly_out:,.0f} ({monthly_out/1e6:.1f} M)")
print(f"Total Monthly Tokens        : {(monthly_in + monthly_out)/1e9:.3f} B")

Expected Input Tokens/query : 32,960.0
Expected Output Tokens/query: 2,889.0
Monthly Queries             : 89,100
Monthly Input Tokens        : 2,936,736,000 (2.937 B)
Monthly Output Tokens       : 257,409,900 (257.4 M)
Total Monthly Tokens        : 3.194 B


## 2. API Cost With and Without Prompt Caching
Prefix length = 1,500 tokens. Evaluated against budget $B = \$1,260$.

In [3]:
# Prefix caching breakdown
prefix_len = 1500
calls_per_query = (N - 1) * (1 + p) + 1  # 7.54 calls
cache_reads_per_query = calls_per_query - 1  # 6.54 cached reads per query

monthly_cache_write = q_month * prefix_len
monthly_cache_read = q_month * (cache_reads_per_query * prefix_len)
monthly_uncached_in = monthly_in - (monthly_cache_write + monthly_cache_read)

print(f"Monthly Uncached Input: {monthly_uncached_in/1e6:,.1f} M")
print(f"Monthly Cache Write   : {monthly_cache_write/1e6:,.1f} M")
print(f"Monthly Cache Read    : {monthly_cache_read/1e6:,.1f} M")

# Models evaluation
models = [
    ("GPT-4o", 2.50, 10.00, 1.25, False),
    ("GPT-4o mini", 0.15, 0.60, 0.075, False),
    ("Claude 3.5 Sonnet", 3.00, 15.00, 0.30, True),
    ("Claude 3.5 Haiku", 0.80, 4.00, 0.08, True),
    ("Gemini 1.5 Flash", 0.075, 0.30, None, False), # min 32k tokens
    ("Llama-3.1-8B (DeepInfra)", 0.055, 0.055, None, False),
    ("Llama-3.3-70B (Together)", 0.88, 0.88, None, False),
    ("Qwen2.5-7B (DeepInfra)", 0.06, 0.06, None, False),
    ("Qwen2.5-72B (DeepInfra)", 0.35, 0.40, None, False),
    ("DeepSeek-V3", 0.14, 0.28, 0.014, False),
]

print(f"{'Model':<25}{'No Cache ($)':>14}{'Cached ($)':>14}{'Within $1,260?':>16}")
print("-" * 70)
for name, p_in, p_out, p_ca, is_anth in models:
    c_no = (monthly_in * p_in + monthly_out * p_out) / 1e6
    if p_ca is not None:
        if is_anth:
            c_ca = (monthly_uncached_in * p_in + monthly_cache_write * (1.25 * p_in) +
                    monthly_cache_read * p_ca + monthly_out * p_out) / 1e6
        else:
            c_ca = (monthly_uncached_in * p_in + monthly_cache_write * p_in +
                    monthly_cache_read * p_ca + monthly_out * p_out) / 1e6
    else:
        c_ca = c_no
    ok = "YES" if c_ca <= B else "NO (Exceeds)"
    print(f"{name:<25}{c_no:>14.2f}{c_ca:>14.2f}{ok:>16}")

Monthly Uncached Input: 1,929.0 M
Monthly Cache Write   : 133.7 M
Monthly Cache Read    : 874.1 M
Model                      No Cache ($)    Cached ($)  Within $1,260?
----------------------------------------------------------------------
GPT-4o                          9915.94       8823.35    NO (Exceeds)
GPT-4o mini                      594.96        529.40             YES
Claude 3.5 Sonnet              12671.36      10411.60    NO (Exceeds)
Claude 3.5 Haiku                3379.03       2776.43    NO (Exceeds)
Gemini 1.5 Flash                 297.48        297.48             YES
Llama-3.1-8B (DeepInfra)         175.68        175.68             YES
Llama-3.3-70B (Together)        2810.85       2810.85    NO (Exceeds)
Qwen2.5-7B (DeepInfra)           191.65        191.65             YES
Qwen2.5-72B (DeepInfra)         1130.82       1130.82             YES
DeepSeek-V3                      483.22        373.08             YES


## 3. Peak Load and Rate Limits (Heatwave Window 12:00–14:00)

In [4]:
peak_queries = 0.35 * (5 * Q)
peak_qpm = peak_queries / 120
peak_rpm = peak_qpm * calls_per_query
peak_in_tpm = peak_qpm * total_exp_in
peak_out_tpm = peak_qpm * total_exp_out
peak_total_tpm = peak_in_tpm + peak_out_tpm

print(f"Heatwave day traffic : {5 * Q:,} queries/day")
print(f"Peak window queries  : {peak_queries:,.1f} queries in 120 mins")
print(f"Peak QPM             : {peak_qpm:.2f} queries/min")
print(f"Peak RPM required    : {peak_rpm:.2f} requests/min")
print(f"Peak TPM required    : {peak_total_tpm:,.0f} tokens/min (Input: {peak_in_tpm:,.0f}, Output: {peak_out_tpm:,.0f})")
print("\nCritical Bottleneck Analysis:")
print("- OpenAI Tier 1 GPT-4o-mini limit = 200,000 TPM -> EXCEEDED by 4.3x! (Requires Tier 2+)")
print("- Anthropic Tier 1 limit = 40,000 TPM -> EXCEEDED by 21x!")
print("- Google Gemini Pay-as-you-go limit = 4,000,000 TPM -> HANDLES PEAK COMFORTABLY")
print("- DeepInfra limit = 600 RPM / 500k TPM -> RPM ok, TPM near ceiling")

Heatwave day traffic : 8,250 queries/day
Peak window queries  : 2,887.5 queries in 120 mins
Peak QPM             : 24.06 queries/min
Peak RPM required    : 181.43 requests/min
Peak TPM required    : 862,617 tokens/min (Input: 793,100, Output: 69,517)

Critical Bottleneck Analysis:
- OpenAI Tier 1 GPT-4o-mini limit = 200,000 TPM -> EXCEEDED by 4.3x! (Requires Tier 2+)
- Anthropic Tier 1 limit = 40,000 TPM -> EXCEEDED by 21x!
- Google Gemini Pay-as-you-go limit = 4,000,000 TPM -> HANDLES PEAK COMFORTABLY
- DeepInfra limit = 600 RPM / 500k TPM -> RPM ok, TPM near ceiling


## 4. Self-Hosting Memory for Llama-3.1-8B-Instruct
Parameters from `config.json`: layers=32, hidden_size=4096, heads=32, kv_heads=8, head_dim=128.

In [5]:
layers, kv_heads, head_dim = 32, 8, 128
max_seq = 7400 # Call 7: 6800 in + 600 out

# Weight memory
mem_fp16 = 8.03e9 * 2 / (1024**3)
mem_int8 = 8.03e9 * 1 / (1024**3)
mem_int4 = 4.05 # GiB

# KV Cache memory per sequence
kv_bytes = 2 * layers * kv_heads * head_dim * max_seq * 2
kv_mib = kv_bytes / (1024**2)
kv_gib = kv_bytes / (1024**3)

# Concurrency on 24GB GPU (RTX 4090 / A10G)
vram_usable = 24.0 * 0.90 # 10% reserve for activations/runtime
vram_for_kv = vram_usable - mem_int4
max_concurrent = int(vram_for_kv * 1024 / kv_mib)

print(f"Weight Memory (FP16) : {mem_fp16:.2f} GiB")
print(f"Weight Memory (INT8) : {mem_int8:.2f} GiB")
print(f"Weight Memory (INT4) : {mem_int4:.2f} GiB")
print(f"KV Cache per seq     : {kv_mib:.2f} MiB ({kv_gib:.3f} GiB)")
print(f"Usable VRAM (24GB)   : {vram_usable:.2f} GiB (after 10% overhead)")
print(f"Max Concurrent Seq   : {max_concurrent} sequences per GPU")

Weight Memory (FP16) : 14.96 GiB
Weight Memory (INT8) : 7.48 GiB
Weight Memory (INT4) : 4.05 GiB
KV Cache per seq     : 925.00 MiB (0.903 GiB)
Usable VRAM (24GB)   : 21.60 GiB (after 10% overhead)
Max Concurrent Seq   : 19 sequences per GPU


## 5. Break-Even Analysis: Cost Step Function vs Daily Volume

In [6]:
import matplotlib.pyplot as plt
import numpy as np

# RunPod RTX 4090 @ $0.44/hr -> $316.80/month
gpu_monthly = 720 * 0.44
cost_per_q_mini = 529.41 / q_month
cost_per_q_gemini = 297.48 / q_month

q_star_mini = gpu_monthly / (54 * cost_per_q_mini)
q_star_gemini = gpu_monthly / (54 * cost_per_q_gemini)

print(f"Cost per GPU / month : ${gpu_monthly:.2f}")
print(f"Break-even Q* vs GPT-4o mini : {q_star_mini:,.1f} queries/day")
print(f"Break-even Q* vs Gemini Flash: {q_star_gemini:,.1f} queries/day")
print(f"Current volume Q = {Q} queries/day -> Self-hosting is CHEAPER than GPT-4o mini!")

Cost per GPU / month : $316.80
Break-even Q* vs GPT-4o mini : 987.4 queries/day
Break-even Q* vs Gemini Flash: 1,757.2 queries/day
Current volume Q = 1650 queries/day -> Self-hosting is CHEAPER than GPT-4o mini!


## 6. Latency Budget and Architectural Redesign
Median tool delay measured empirically: Open-Meteo Air Quality ~1.00s, Weather ~0.98s, local tools ~0.001ms.

In [7]:
# Measured median tool delay
median_tool_s = 0.985

# Sequential execution: 6 intermediate calls (350 tok) + 1 final call (600 tok) + retries
def seq_latency(ttft, tps):
    t_llm = (1 + p) * (N - 1) * (ttft + 350 / tps) + (ttft + 600 / tps)
    t_tools = (1 + p) * (N - 1) * median_tool_s
    return t_llm + t_tools, t_llm, t_tools

lat_gpt, llm_g, tool_g = seq_latency(0.28, 120.0)
lat_llama, llm_l, tool_l = seq_latency(0.18, 140.0)

print(f"Sequential GPT-4o mini latency : {lat_gpt:.2f} s (LLM: {llm_g:.2f}s, Tools: {tool_g:.2f}s) -> Target T={T:.1f}s EXCEEDED!")
print(f"Sequential Llama-3.1-8B latency: {lat_llama:.2f} s (LLM: {llm_l:.2f}s, Tools: {tool_l:.2f}s) -> Target T={T:.1f}s EXCEEDED!")

# Redesigned architecture:
# Step 1: Router & intent extraction (100 tokens)
# Step 2: Parallel fetch of Air Quality & Weather + local Heat Index & KB lookup (~1.0s)
# Step 3: Streamed response generation (600 tokens)
def opt_latency(ttft, tps):
    t_step1 = ttft + 100 / tps
    t_tools = median_tool_s # parallel network calls
    t_step2 = ttft + 600 / tps
    t_first_token = t_step1 + t_tools + ttft
    return t_step1 + t_tools + t_step2, t_first_token

lat_opt_gpt, ttft_stream_gpt = opt_latency(0.28, 120.0)
lat_opt_llama, ttft_stream_llama = opt_latency(0.18, 140.0)

print(f"\n--- Optimized Architecture (Parallel Tools + Streaming) ---")
print(f"Optimized GPT-4o mini  : {lat_opt_gpt:.2f} s total (Perceived TTFT: {ttft_stream_gpt:.2f} s) -> MEETS T={T:.1f}s!")
print(f"Optimized Llama-3.1-8B : {lat_opt_llama:.2f} s total (Perceived TTFT: {ttft_stream_llama:.2f} s) -> MEETS T={T:.1f}s!")

Sequential GPT-4o mini latency : 32.63 s (LLM: 26.19s, Tools: 6.44s) -> Target T=10.0s EXCEEDED!
Sequential Llama-3.1-8B latency: 28.43 s (LLM: 21.99s, Tools: 6.44s) -> Target T=10.0s EXCEEDED!

--- Optimized Architecture (Parallel Tools + Streaming) ---
Optimized GPT-4o mini  : 7.38 s total (Perceived TTFT: 2.38 s) -> MEETS T=10.0s!
Optimized Llama-3.1-8B : 6.34 s total (Perceived TTFT: 2.06 s) -> MEETS T=10.0s!
